# Agent 3 — Silver SQL Review Agent

Reviews Silver transformation logic for risky SQL patterns
and validates actual row count ratios against expected survival rates.

On-demand review triggered by a developer submitting
a transformation for review.

Checks implemented:
  Check 1:  Join Fan-out Detection
  Check 2:  Missing Null Filter Detection
  Check 3:  Aggregation Before Filter Detection
  Check 10: Row Count Ratio Validation

Reads from:
  uc3_transformation_registry_b
  uc2_layer_registry_b
  Raw and Silver Delta tables
  Notebook source via Workspace API

Writes to:
  uc3_review_results_b
  uc3_transformation_registry_b (updates last_review fields)

## Parameters

In [0]:
dbutils.widgets.text(
    "transformation_id",
    "",
    "Transformation ID (leave blank to review all active)"
)

dbutils.widgets.text(
    "business_date",
    "",
    "Business Date for row count check (YYYY-MM-DD, blank = yesterday)"
)

TRANSFORMATION_ID_FILTER = dbutils.widgets.get("transformation_id").strip()
BUSINESS_DATE_STR        = dbutils.widgets.get("business_date").strip()

from datetime import date, datetime, timedelta

if not BUSINESS_DATE_STR:
    BUSINESS_DATE = date.today() - timedelta(days=1)
    date_source   = "default (yesterday)"
else:
    try:
        BUSINESS_DATE = date.fromisoformat(BUSINESS_DATE_STR)
        date_source   = "provided parameter"
    except ValueError:
        BUSINESS_DATE = date.today() - timedelta(days=1)
        date_source   = "fallback (invalid format)"

print("=" * 60)
print("AGENT 3 — SILVER SQL REVIEW AGENT")
print("=" * 60)
print(f"Transformation filter : {TRANSFORMATION_ID_FILTER or 'All active'}")
print(f"Business date         : {BUSINESS_DATE}  ({date_source})")
print(f"Started at            : {datetime.now()}")
print("=" * 60)

In [0]:
import re
import uuid
import json
import base64
import requests
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

TRANSFORMATION_REGISTRY = f"{CATALOG}.{SCHEMA}.uc3_transformation_registry_b"
REVIEW_RESULTS          = f"{CATALOG}.{SCHEMA}.uc3_review_results_b"
LAYER_REGISTRY          = f"{CATALOG}.{SCHEMA}.uc2_layer_registry_b"

# ── Workspace API ──────────────────────────────────────────────────────────

HOST  = spark.conf.get("spark.databricks.workspaceUrl")
TOKEN = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type" : "application/json"
}

# ── Severity Levels ────────────────────────────────────────────────────────

STATUS_PASS     = "PASS"
STATUS_WARNING  = "WARNING"
STATUS_CRITICAL = "CRITICAL"
STATUS_ERROR    = "ERROR"

SEVERITY_RANK = {
    STATUS_PASS    : 0,
    STATUS_WARNING : 1,
    STATUS_CRITICAL: 2,
    STATUS_ERROR   : 3
}

# ── AI Confidence Threshold ────────────────────────────────────────────────

AI_MIN_FAILED_CHECKS  = 1    # Agent 3 triggers AI on any failure
AI_MIN_CRITICAL_COUNT = 0    # Even warnings justify explanation for SQL review

print("Configuration loaded.")

## Step 1: Read Notebook Source Code

In [0]:
def read_notebook_source(notebook_path: str) -> str:
    """
    Exports a notebook from the Databricks workspace as Python source.
    Returns the full source text of the notebook.
    """
    response = requests.get(
        f"https://{HOST}/api/2.0/workspace/export",
        headers=HEADERS,
        params={
            "path"  : notebook_path,
            "format": "SOURCE"
        }
    )

    if response.status_code != 200:
        logger.error(
            f"Failed to read notebook at {notebook_path}. "
            f"Status: {response.status_code}. "
            f"Error: {response.text}"
        )
        return ""

    content_b64 = response.json().get("content", "")
    if not content_b64:
        logger.warning(f"Empty content returned for {notebook_path}")
        return ""

    return base64.b64decode(content_b64).decode("utf-8")


def extract_function_source(
    notebook_source : str,
    function_name   : str
) -> str:
    """
    Extracts the source code of a specific Python function
    from the notebook source text.

    Starts at the function definition line.
    Ends when the next top-level function or class definition is found.
    """
    if not function_name or not notebook_source:
        return notebook_source

    lines         = notebook_source.split("\n")
    start_line    = None
    end_line      = None
    in_function   = False

    for i, line in enumerate(lines):
        # Detect function start
        if f"def {function_name}(" in line:
            start_line  = i
            in_function = True
            continue

        # Detect end: next top-level def or class
        if in_function and i > (start_line or 0):
            stripped = line.strip()
            if (
                stripped.startswith("def ")
                or stripped.startswith("class ")
                or stripped.startswith("# COMMAND ----------")
            ) and not line.startswith(" ") and not line.startswith("\t"):
                end_line = i
                break

    if start_line is None:
        logger.warning(
            f"Function '{function_name}' not found in notebook source."
        )
        return notebook_source

    if end_line is None:
        end_line = len(lines)

    function_lines = lines[start_line:end_line]
    return "\n".join(function_lines)


print("Notebook source reader loaded.")

## Step 2: Static Analysis Check Implementations

In [0]:
def check_join_fanout(
    source_code      : str,
    transformation_id: str,
    target_fqn       : str
) -> dict:
    """
    Check 1: Join Fan-out Detection

    Detects join patterns that could produce more output rows
    than input rows. In PySpark this appears as:
      - .join() calls without a subsequent .dropDuplicates()
      - join() on a column that is not guaranteed to be unique
        in the right DataFrame

    Also detects SQL JOIN patterns where the right table
    is not deduplicated before joining.

    Severity:
      WARNING  if a join exists without obvious deduplication
      CRITICAL if a join is on a non-key column with no dedup
    """
    t_start  = datetime.now()
    findings = []

    # Pattern 1: PySpark .join() calls
    join_pattern   = re.compile(r'\.join\s*\(', re.MULTILINE)
    dedup_pattern  = re.compile(
        r'\.dropDuplicates\s*\(|\.distinct\s*\(', re.MULTILINE
    )

    join_matches  = list(join_pattern.finditer(source_code))
    dedup_matches = list(dedup_pattern.finditer(source_code))

    if join_matches:
        join_count  = len(join_matches)
        dedup_count = len(dedup_matches)

        findings.append(
            f"Found {join_count} .join() call(s) in transformation. "
            f"Found {dedup_count} .dropDuplicates()/.distinct() call(s)."
        )

        if dedup_count == 0:
            status = STATUS_WARNING
            findings.append(
                "WARNING: No deduplication found after join operations. "
                "Verify that the right-side DataFrame has unique keys "
                "to prevent row multiplication."
            )
        else:
            status = STATUS_PASS
            findings.append(
                "Deduplication present. Verify it occurs AFTER joins "
                "to prevent fan-out from reaching output."
            )
    else:
        status = STATUS_PASS
        findings.append("No join operations detected in transformation.")

    # Pattern 2: SQL JOIN keyword detection
    sql_join_pattern = re.compile(
        r'\bJOIN\b', re.MULTILINE | re.IGNORECASE
    )
    sql_joins = list(sql_join_pattern.finditer(source_code))

    if sql_joins:
        findings.append(
            f"Found {len(sql_joins)} SQL JOIN keyword(s). "
            "Verify right-side tables have unique join keys."
        )
        if status == STATUS_PASS:
            status = STATUS_WARNING

    duration = (datetime.now() - t_start).total_seconds()

    return {
        "check_name"     : "JOIN_FANOUT",
        "check_status"   : status,
        "check_level"    : "STATIC",
        "findings"       : findings,
        "pattern_matches": {
            "pyspark_joins"         : len(join_matches),
            "deduplication_calls"   : len(dedup_matches),
            "sql_join_keywords"     : len(sql_joins)
        },
        "duration_seconds": duration,
        "recommendation" : (
            "Ensure right-side DataFrames are deduplicated before joining. "
            "Add .dropDuplicates([join_key]) to the right DataFrame "
            "if uniqueness is not guaranteed by the source."
            if status != STATUS_PASS else "No action required."
        )
    }


def check_null_filter(
    source_code      : str,
    transformation_id: str,
    target_fqn       : str
) -> dict:
    """
    Check 2: Missing Null Filter Detection

    Detects when columns used in joins or key operations
    are not protected against null values.

    In PySpark null handling appears as:
      - .filter(F.col("x").isNotNull())
      - .na.drop(subset=["x"])
      - .dropna()
      - .fillna()
      - F.coalesce()

    Absence of null handling on join keys is a WARNING.
    Absence on non-nullable columns is CRITICAL.
    """
    t_start  = datetime.now()
    findings = []

    # Check for null handling patterns
    null_patterns = {
        "isNotNull()"   : re.compile(r'\.isNotNull\s*\(\s*\)', re.MULTILINE),
        "isNull()"      : re.compile(r'\.isNull\s*\(\s*\)', re.MULTILINE),
        "na.drop()"     : re.compile(r'\.na\.drop\s*\(', re.MULTILINE),
        "dropna()"      : re.compile(r'\.dropna\s*\(', re.MULTILINE),
        "fillna()"      : re.compile(r'\.fillna\s*\(', re.MULTILINE),
        "coalesce()"    : re.compile(r'F\.coalesce\s*\(|coalesce\s*\(', re.MULTILINE),
        "IS NOT NULL"   : re.compile(r'IS\s+NOT\s+NULL', re.MULTILINE | re.IGNORECASE),
        "IS NULL"       : re.compile(r'IS\s+NULL', re.MULTILINE | re.IGNORECASE)
    }

    null_handling_found = {}
    total_null_handling = 0

    for pattern_name, pattern in null_patterns.items():
        matches = list(pattern.finditer(source_code))
        if matches:
            null_handling_found[pattern_name] = len(matches)
            total_null_handling += len(matches)

    findings.append(
        f"Null handling patterns found: {total_null_handling} total"
    )
    for name, count in null_handling_found.items():
        findings.append(f"  {name}: {count} occurrence(s)")

    # Check for join operations without null guards
    join_pattern = re.compile(r'\.join\s*\(', re.MULTILINE)
    join_matches = list(join_pattern.finditer(source_code))

    if join_matches and total_null_handling == 0:
        status = STATUS_CRITICAL
        findings.append(
            "CRITICAL: Join operations found but no null handling detected. "
            "Null values in join keys will silently drop records."
        )
    elif join_matches and total_null_handling < len(join_matches):
        status = STATUS_WARNING
        findings.append(
            f"WARNING: {len(join_matches)} join(s) found but only "
            f"{total_null_handling} null handling pattern(s). "
            "Some join keys may not be null-protected."
        )
    elif total_null_handling == 0:
        status = STATUS_WARNING
        findings.append(
            "WARNING: No null handling detected in transformation. "
            "Verify that all key columns are guaranteed non-null "
            "by upstream processes."
        )
    else:
        status = STATUS_PASS
        findings.append(
            "Null handling patterns are present in the transformation."
        )

    duration = (datetime.now() - t_start).total_seconds()

    return {
        "check_name"       : "NULL_FILTER",
        "check_status"     : status,
        "check_level"      : "STATIC",
        "findings"         : findings,
        "null_patterns"    : null_handling_found,
        "join_count"       : len(join_matches),
        "duration_seconds" : duration,
        "recommendation"   : (
            "Add .filter(F.col('key_column').isNotNull()) before join operations. "
            "Use F.coalesce() for columns that may be null but should have defaults."
            if status != STATUS_PASS else "No action required."
        )
    }


def check_aggregation_order(
    source_code      : str,
    transformation_id: str,
    target_fqn       : str
) -> dict:
    """
    Check 3: Aggregation Before Filter Detection

    Detects when aggregation (groupBy/agg) appears before
    filtering operations that should be applied first.

    This is both a performance issue (aggregating more rows
    than necessary) and potentially a logic issue (aggregating
    data that should have been excluded).

    Pattern to detect:
      .groupBy(...).agg(...)   followed later by   .filter(...)
    """
    t_start  = datetime.now()
    findings = []

    lines = source_code.split("\n")

    # Find line numbers of key operations
    groupby_lines = []
    agg_lines     = []
    filter_lines  = []

    for i, line in enumerate(lines):
        stripped = line.strip()
        if ".groupBy(" in stripped or ".groupby(" in stripped:
            groupby_lines.append(i + 1)  # 1-indexed line numbers
        if ".agg(" in stripped:
            agg_lines.append(i + 1)
        if (
            ".filter(" in stripped or
            "WHERE " in stripped.upper() or
            ".where(" in stripped
        ):
            filter_lines.append(i + 1)

    findings.append(
        f"groupBy() calls found at lines: {groupby_lines}"
    )
    findings.append(
        f"agg() calls found at lines: {agg_lines}"
    )
    findings.append(
        f"filter/where calls found at lines: {filter_lines}"
    )

    # Check if any filter comes AFTER a groupBy
    violations = []

    for agg_line in agg_lines:
        for filter_line in filter_lines:
            if filter_line > agg_line:
                violations.append({
                    "agg_at_line"   : agg_line,
                    "filter_at_line": filter_line
                })

    if violations:
        status = STATUS_WARNING
        findings.append(
            f"WARNING: Found {len(violations)} case(s) where filter() "
            f"appears after agg(). "
            f"Consider moving filters before aggregation to reduce "
            f"the data volume being aggregated."
        )
        for v in violations:
            findings.append(
                f"  agg() at line {v['agg_at_line']} → "
                f"filter() at line {v['filter_at_line']}"
            )
    else:
        status = STATUS_PASS
        findings.append(
            "No aggregation-before-filter patterns detected."
        )

    # Check for HAVING clause (SQL aggregation filter)
    having_pattern = re.compile(r'\bHAVING\b', re.MULTILINE | re.IGNORECASE)
    having_matches = list(having_pattern.finditer(source_code))

    if having_matches:
        findings.append(
            f"Found {len(having_matches)} HAVING clause(s). "
            "Verify these filters cannot be applied as WHERE clauses "
            "before aggregation for better performance."
        )
        if status == STATUS_PASS:
            status = STATUS_WARNING

    duration = (datetime.now() - t_start).total_seconds()

    return {
        "check_name"      : "AGGREGATION_ORDER",
        "check_status"    : status,
        "check_level"     : "STATIC",
        "findings"        : findings,
        "groupby_lines"   : groupby_lines,
        "agg_lines"       : agg_lines,
        "filter_lines"    : filter_lines,
        "violations"      : violations,
        "duration_seconds": duration,
        "recommendation"  : (
            "Move .filter() calls before .groupBy().agg() where possible. "
            "Apply all row-level filters before aggregating to reduce "
            "the data volume processed by the aggregation stage."
            if status != STATUS_PASS else "No action required."
        )
    }


def check_row_count_ratio(
    transformation_id   : str,
    target_fqn          : str,
    source_fqn          : str,
    business_date       : date,
    expected_survival   : float
) -> dict:
    """
    Check 10: Row Count Ratio Validation

    Compares actual row count ratio between Raw and Silver
    against the expected survival rate configured in the
    transformation registry.

    Detects:
      - Fan-out: Silver has MORE rows than Raw (join multiplication)
      - Over-filtering: Silver has MUCH FEWER rows than expected
      - Under-filtering: Silver has MORE rows than expected
    """
    t_start  = datetime.now()
    findings = []

    # Determine date column for source (raw) table
    source_date_col = "transaction_date" if "transaction" in source_fqn else "business_date"
    target_date_col = "transaction_date" if "transaction" in target_fqn else "business_date"

    try:
        # Count raw (source) records for this business date
        source_df    = spark.table(source_fqn)
        source_count = source_df.filter(
            F.col(source_date_col) == F.lit(str(business_date))
        ).count()

        # Count silver (target) records for this business date
        target_df    = spark.table(target_fqn)
        target_count = target_df.filter(
            F.col(target_date_col) == F.lit(str(business_date))
        ).count()

        if source_count == 0:
            return {
                "check_name"     : "ROW_COUNT_RATIO",
                "check_status"   : STATUS_WARNING,
                "check_level"    : "RUNTIME",
                "findings"       : [
                    f"No source records found for {business_date}. "
                    "Cannot compute row count ratio."
                ],
                "duration_seconds": (datetime.now() - t_start).total_seconds()
            }

        if target_count == 0:
            return {
                "check_name"     : "ROW_COUNT_RATIO",
                "check_status"   : STATUS_CRITICAL,
                "check_level"    : "RUNTIME",
                "findings"       : [
                    f"CRITICAL: Silver table has ZERO records for {business_date} "
                    f"while Raw has {source_count:,} records. "
                    "Silver transformation may have failed entirely."
                ],
                "source_count"   : source_count,
                "target_count"   : target_count,
                "actual_survival": 0.0,
                "expected_survival": expected_survival,
                "duration_seconds": (datetime.now() - t_start).total_seconds()
            }

        actual_survival   = target_count / source_count
        deviation         = actual_survival - expected_survival
        deviation_pct     = abs(deviation) / expected_survival * 100

        findings.append(
            f"Source (Raw) row count  : {source_count:,}"
        )
        findings.append(
            f"Target (Silver) row count: {target_count:,}"
        )
        findings.append(
            f"Actual survival rate    : {actual_survival:.2%}"
        )
        findings.append(
            f"Expected survival rate  : {expected_survival:.2%}"
        )
        findings.append(
            f"Deviation               : {deviation:+.2%} "
            f"({deviation_pct:.1f}% from expected)"
        )

        # Detect fan-out: Silver > Raw
        if actual_survival > 1.05:
            status = STATUS_CRITICAL
            findings.append(
                f"CRITICAL: Silver has MORE records than Raw "
                f"({actual_survival:.2%} survival). "
                "This indicates a JOIN fan-out producing duplicate rows."
            )
        # Detect severe over-filtering
        elif deviation < -0.15:
            status = STATUS_CRITICAL
            findings.append(
                f"CRITICAL: Survival rate {actual_survival:.2%} is "
                f"{abs(deviation):.2%} below expected {expected_survival:.2%}. "
                "Transformation may be over-filtering valid records."
            )
        # Detect moderate deviation
        elif deviation_pct > 5.0:
            status = STATUS_WARNING
            findings.append(
                f"WARNING: Survival rate {actual_survival:.2%} deviates "
                f"{deviation_pct:.1f}% from expected {expected_survival:.2%}. "
                "Monitor for trend."
            )
        else:
            status = STATUS_PASS
            findings.append(
                f"Survival rate is within acceptable range of expected."
            )

        duration = (datetime.now() - t_start).total_seconds()

        return {
            "check_name"        : "ROW_COUNT_RATIO",
            "check_status"      : status,
            "check_level"       : "RUNTIME",
            "findings"          : findings,
            "source_count"      : source_count,
            "target_count"      : target_count,
            "actual_survival"   : round(actual_survival, 6),
            "expected_survival" : expected_survival,
            "deviation_pct"     : round(deviation_pct, 4),
            "business_date"     : str(business_date),
            "duration_seconds"  : duration,
            "recommendation"    : (
                "Investigate the transformation logic for JOIN fan-out or "
                "incorrect filter conditions causing unexpected row counts."
                if status != STATUS_PASS else "No action required."
            )
        }

    except Exception as e:
        logger.error(f"Row count ratio check failed: {str(e)}")
        return {
            "check_name"     : "ROW_COUNT_RATIO",
            "check_status"   : STATUS_ERROR,
            "check_level"    : "RUNTIME",
            "findings"       : [f"Check failed with error: {str(e)}"],
            "duration_seconds": (datetime.now() - t_start).total_seconds()
        }


print("Static and runtime check functions loaded.")

## Step 3: Score Results and Build Review Record

In [0]:
def determine_overall_status(check_results: list) -> str:
    """
    Returns the worst status across all check results.
    """
    worst = STATUS_PASS
    for result in check_results:
        check_status = result.get("check_status", STATUS_PASS)
        if SEVERITY_RANK.get(check_status, 0) > SEVERITY_RANK.get(worst, 0):
            worst = check_status
    return worst


def build_review_record(
    review_id      : str,
    transformation : dict,
    check_results  : list,
    overall_status : str,
    business_date  : date
) -> dict:
    """
    Builds the review result record ready for writing to uc3_review_results_b.
    """
    return {
        "review_id"                  : review_id,
        "transformation_id"          : transformation["transformation_id"],
        "target_fully_qualified_name": transformation["target_fully_qualified_name"],
        "review_timestamp"           : datetime.now(),
        "triggered_by"               : "MANUAL",
        "business_date"              : business_date,
        "overall_status"             : overall_status,
        "total_checks_run"           : len(check_results),
        "total_checks_passed"        : sum(
            1 for r in check_results
            if r.get("check_status") == STATUS_PASS
        ),
        "total_checks_warned"        : sum(
            1 for r in check_results
            if r.get("check_status") == STATUS_WARNING
        ),
        "total_checks_critical"      : sum(
            1 for r in check_results
            if r.get("check_status") == STATUS_CRITICAL
        ),
        "check_results"              : json.dumps(check_results),
        "ai_summary"                 : None,
        "ai_recommended_action"      : None,
        "ai_failure_category"        : None,
        "ai_confidence_score"        : None,
        "ai_model_version"           : None,
        "ai_invocation_id"           : None,
        "alert_sent"                 : None,
        "alert_sent_timestamp"       : None,
        "alert_channel"              : None,
        "created_timestamp"          : datetime.now(),
        "created_by"                 : "01_sql_review_agent"
    }


print("Scoring and results builder loaded.")

## Step 4: Write Review Results

In [0]:
def write_review_result(review_record: dict) -> bool:
    """
    Writes a single review result to uc3_review_results_b.
    """
    from pyspark.sql.types import (
        StructType, StructField, StringType,
        TimestampType, IntegerType, BooleanType,
        DoubleType, DateType
    )

    schema = StructType([
        StructField("review_id",                   StringType(),    False),
        StructField("transformation_id",            StringType(),    False),
        StructField("target_fully_qualified_name",  StringType(),    False),
        StructField("review_timestamp",             TimestampType(), False),
        StructField("triggered_by",                 StringType(),    True),
        StructField("business_date",                DateType(),      True),
        StructField("overall_status",               StringType(),    False),
        StructField("total_checks_run",             IntegerType(),   False),
        StructField("total_checks_passed",          IntegerType(),   False),
        StructField("total_checks_warned",          IntegerType(),   False),
        StructField("total_checks_critical",        IntegerType(),   False),
        StructField("check_results",                StringType(),    True),
        StructField("ai_summary",                   StringType(),    True),
        StructField("ai_recommended_action",        StringType(),    True),
        StructField("ai_failure_category",          StringType(),    True),
        StructField("ai_confidence_score",          DoubleType(),    True),
        StructField("ai_model_version",             StringType(),    True),
        StructField("ai_invocation_id",             StringType(),    True),
        StructField("alert_sent",                   BooleanType(),   True),
        StructField("alert_sent_timestamp",         TimestampType(), True),
        StructField("alert_channel",                StringType(),    True),
        StructField("created_timestamp",            TimestampType(), False),
        StructField("created_by",                   StringType(),    False),
    ])

    col_order = [f.name for f in schema.fields]
    row       = [tuple(review_record.get(col) for col in col_order)]
    df        = spark.createDataFrame(row, schema=schema)

    df.write.mode("append").saveAsTable(REVIEW_RESULTS)
    print(f"  Written review result to {REVIEW_RESULTS}")
    return True


def update_transformation_last_review(
    transformation_id: str,
    status           : str
):
    """
    Updates the last_review fields in uc3_transformation_registry_b.
    """
    spark.sql(f"""
        UPDATE {TRANSFORMATION_REGISTRY}
        SET
            last_reviewed_timestamp = '{datetime.now()}',
            last_review_status      = '{status}'
        WHERE transformation_id = '{transformation_id}'
    """)
    print(f"  Updated last_review_status to {status} for {transformation_id}")


print("Write functions loaded.")

## Step 5: Run Reviews for All Active Transformations

In [0]:
# ── Load transformations to review ─────────────────────────────────────────

df = spark.table(TRANSFORMATION_REGISTRY).filter(
    F.col("is_active") == True
)

if TRANSFORMATION_ID_FILTER:
    df = df.filter(
        F.col("transformation_id") == TRANSFORMATION_ID_FILTER
    )

transformations = [row.asDict() for row in df.collect()]

print(f"Transformations to review: {len(transformations)}")
for t in transformations:
    print(
        f"  {t['transformation_id']:<25} "
        f"{t['target_fully_qualified_name'].split('.')[-1]}"
    )

all_review_records = []
reviews_done       = 0
reviews_failed     = 0

print()

for txfm in transformations:

    txfm_id    = txfm["transformation_id"]
    target_fqn = txfm["target_fully_qualified_name"]
    source_fqn = txfm["source_fully_qualified_name"]
    nb_path    = txfm["notebook_path"]
    func_name  = txfm["function_name"]
    survival   = float(txfm.get("expected_survival_rate") or 0.92)
    review_id  = str(uuid.uuid4())

    print(f"\n{'='*60}")
    print(f"Transformation : {txfm_id}")
    print(f"Target table   : {target_fqn.split('.')[-1]}")
    print(f"Function       : {func_name}")
    print(f"Notebook       : {nb_path}")
    print(f"Review ID      : {review_id}")
    print(f"{'='*60}")

    check_results = []

    try:
        # ── Read notebook source ───────────────────────────────────────────
        print(f"\n  Reading notebook source...")
        full_source = read_notebook_source(nb_path)

        if not full_source:
            print(f"  Could not read notebook source. Skipping.")
            reviews_failed += 1
            continue

        print(f"  Source length: {len(full_source):,} characters")

        # ── Extract function source ────────────────────────────────────────
        if func_name:
            print(f"  Extracting function: {func_name}")
            source_to_analyze = extract_function_source(full_source, func_name)
            print(
                f"  Function source: {len(source_to_analyze):,} characters"
            )
        else:
            source_to_analyze = full_source

        # ── Run Check 1: Join Fan-out ──────────────────────────────────────
        print(f"\n  Running Check 1: JOIN_FANOUT")
        result_1 = check_join_fanout(source_to_analyze, txfm_id, target_fqn)
        check_results.append(result_1)
        print(f"  Status: {result_1['check_status']}")

        # ── Run Check 2: Null Filter ───────────────────────────────────────
        print(f"\n  Running Check 2: NULL_FILTER")
        result_2 = check_null_filter(source_to_analyze, txfm_id, target_fqn)
        check_results.append(result_2)
        print(f"  Status: {result_2['check_status']}")

        # ── Run Check 3: Aggregation Order ────────────────────────────────
        print(f"\n  Running Check 3: AGGREGATION_ORDER")
        result_3 = check_aggregation_order(
            source_to_analyze, txfm_id, target_fqn
        )
        check_results.append(result_3)
        print(f"  Status: {result_3['check_status']}")

        # ── Run Check 10: Row Count Ratio ─────────────────────────────────
        print(f"\n  Running Check 10: ROW_COUNT_RATIO")
        result_10 = check_row_count_ratio(
            txfm_id, target_fqn, source_fqn, BUSINESS_DATE, survival
        )
        check_results.append(result_10)
        print(f"  Status: {result_10['check_status']}")

        # ── Score overall ──────────────────────────────────────────────────
        overall_status = determine_overall_status(check_results)

        # ── Build and write review record ──────────────────────────────────
        review_record = build_review_record(
            review_id, txfm, check_results, overall_status,
            business_date=BUSINESS_DATE
        )
        write_review_result(review_record)
        update_transformation_last_review(txfm_id, overall_status)

        all_review_records.append(review_record)
        reviews_done += 1

        # ── Print summary ──────────────────────────────────────────────────
        checks_passed  = sum(
            1 for r in check_results if r["check_status"] == STATUS_PASS
        )
        checks_warned  = sum(
            1 for r in check_results if r["check_status"] == STATUS_WARNING
        )
        checks_critical = sum(
            1 for r in check_results if r["check_status"] == STATUS_CRITICAL
        )

        print(f"\n  {'─'*50}")
        print(f"  REVIEW SUMMARY: {target_fqn.split('.')[-1]}")
        print(f"  {'─'*50}")
        print(f"  Overall status  : {overall_status}")
        print(f"  Checks run      : {len(check_results)}")
        print(f"  Passed          : {checks_passed}")
        print(f"  Warning         : {checks_warned}")
        print(f"  Critical        : {checks_critical}")
        print(f"  {'─'*50}")

    except Exception as e:
        print(f"\n  UNEXPECTED ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        reviews_failed += 1

print(f"\n\nAll transformations reviewed.")
print(f"Done   : {reviews_done}")
print(f"Failed : {reviews_failed}")

## Step 6: AI Summarization for Failed Reviews

In [0]:
MODEL_ENDPOINT = "databricks-gpt-oss-20b"
MODEL_VERSION  = "gpt-oss-20b-080525"
MAX_TOKENS     = 4096
TEMPERATURE    = 0.1
MODEL_URL      = f"https://{HOST}/serving-endpoints/{MODEL_ENDPOINT}/invocations"

VALID_CATEGORIES = {
    "LOGIC_DEFECT",
    "PERFORMANCE_ANTIPATTERN",
    "DATA_QUALITY_RISK",
    "SCHEMA_RISK",
    "COMBINED_ISSUE"
}


def build_agent3_prompt(review_record: dict, txfm: dict) -> str:
    """
    Builds the AI prompt for Agent 3 SQL review summarization.
    """
    target        = review_record["target_fully_qualified_name"]
    table_name    = target.split(".")[-1]
    func_name     = txfm.get("function_name", "transformation")
    description   = txfm.get("transformation_description", "")
    check_results = json.loads(review_record.get("check_results", "[]"))

    failed_checks = [
        r for r in check_results
        if r.get("check_status") in (STATUS_WARNING, STATUS_CRITICAL)
    ]

    checks_text = ""
    for i, check in enumerate(failed_checks, 1):
        checks_text += f"""
  Check {i}: {check['check_name']}
    Status  : {check['check_status']}
    Level   : {check['check_level']}
    Findings: {' | '.join(check.get('findings', []))}
    Action  : {check.get('recommendation', 'N/A')}
"""

    return f"""You are a senior data engineering reviewer analyzing
Silver transformation code for a retail data platform.

TRANSFORMATION BEING REVIEWED:
  Function name : {func_name}
  Target table  : {table_name}
  Description   : {description}

FAILED CHECKS ({len(failed_checks)} issues found):
{checks_text}

Produce a JSON response with exactly these four fields:

{{
  "ai_summary": "2-3 sentences explaining what code quality
                  issues were found in the {func_name} transformation
                  and why they are risky for the Silver layer.",

  "ai_recommended_action": "1-2 specific code changes the developer
                             should make. Reference the specific
                             function name and pattern to fix.",

  "ai_failure_category": "One of: LOGIC_DEFECT, PERFORMANCE_ANTIPATTERN,
                           DATA_QUALITY_RISK, SCHEMA_RISK, COMBINED_ISSUE",

  "ai_confidence_score": 0.0 to 1.0
}}

Respond with valid JSON only."""


def call_agent3_model(prompt: str) -> dict:
    """Calls the AI model for Agent 3 review summarization."""
    payload = {
        "messages": [
            {
                "role"   : "system",
                "content": (
                    "You are a senior data engineer reviewing "
                    "Silver transformation code. "
                    "Respond with valid JSON only."
                )
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        "max_tokens" : MAX_TOKENS,
        "temperature": TEMPERATURE
    }

    try:
        response = requests.post(
            MODEL_URL, headers=HEADERS, json=payload, timeout=120
        )

        if response.status_code != 200:
            return {"success": False, "error": f"HTTP {response.status_code}"}

        data    = response.json()
        choices = data.get("choices", [])
        if not choices:
            return {"success": False, "error": "No choices in response"}

        message = choices[0].get("message", {})
        content = message.get("content", "")

        # Handle list content (reasoning model format)
        if isinstance(content, list):
            text_parts = []
            for block in content:
                if isinstance(block, dict):
                    if block.get("type") == "text":
                        text_parts.append(block.get("text", ""))
                    elif block.get("type") == "reasoning":
                        for s in block.get("summary", []):
                            if isinstance(s, dict):
                                text_parts.append(s.get("text", ""))
            content = " ".join(text_parts)

        if not isinstance(content, str):
            content = str(content)

        # Parse JSON
        start = content.find("{")
        end   = content.rfind("}") + 1

        if start == -1 or end == 0:
            return {"success": False, "error": "No JSON in response"}

        parsed = json.loads(content[start:end])

        category = str(parsed.get("ai_failure_category", "")).upper()
        if category not in VALID_CATEGORIES:
            category = "COMBINED_ISSUE"

        try:
            confidence = float(parsed.get("ai_confidence_score", 0.7))
            confidence = max(0.0, min(1.0, confidence))
        except (TypeError, ValueError):
            confidence = 0.7

        return {
            "success"              : True,
            "ai_summary"           : str(parsed.get("ai_summary", "")),
            "ai_recommended_action": str(parsed.get("ai_recommended_action", "")),
            "ai_failure_category"  : category,
            "ai_confidence_score"  : confidence
        }

    except Exception as e:
        return {"success": False, "error": str(e)}


# ── Process reviews that need AI enrichment ────────────────────────────────

print("Processing AI enrichment for failed reviews...")
print()

ai_enriched = 0

for i, review_record in enumerate(all_review_records):
    overall = review_record.get("overall_status", STATUS_PASS)

    if overall == STATUS_PASS:
        print(
            f"  Skipping {review_record['target_fully_qualified_name'].split('.')[-1]} "
            f"(PASS — no AI needed)"
        )
        continue

    txfm = next(
        (t for t in transformations
         if t["transformation_id"] == review_record["transformation_id"]),
        {}
    )

    print(
        f"  AI enriching: "
        f"{review_record['target_fully_qualified_name'].split('.')[-1]}"
    )

    invocation_id = str(uuid.uuid4())
    prompt        = build_agent3_prompt(review_record, txfm)
    ai_result     = call_agent3_model(prompt)

    if ai_result["success"]:
        enriched_at = datetime.now()

        spark.sql(f"""
            UPDATE {REVIEW_RESULTS}
            SET
                ai_summary              = '{ai_result['ai_summary'].replace("'", "\\'")}',
                ai_recommended_action   = '{ai_result['ai_recommended_action'].replace("'", "\\'")}',
                ai_failure_category     = '{ai_result['ai_failure_category']}',
                ai_confidence_score     = {ai_result['ai_confidence_score']},
                ai_model_version        = '{MODEL_VERSION}',
                ai_invocation_id        = '{invocation_id}'
            WHERE review_id = '{review_record['review_id']}'
        """)

        review_record.update(ai_result)
        review_record["ai_invocation_id"] = invocation_id
        ai_enriched += 1
        print(
            f"  AI complete: "
            f"{ai_result['ai_failure_category']} "
            f"confidence={ai_result['ai_confidence_score']:.2f}"
        )
    else:
        print(f"  AI failed: {ai_result.get('error')}")

print(f"\nAI enrichment complete. {ai_enriched} reviews enriched.")

## Step 7: View Review Results

In [0]:
print("=" * 60)
print("AGENT 3 REVIEW RESULTS")
print("=" * 60)

# Review summary
print("\n1. Review Summary:\n")
spark.sql(f"""
    SELECT
        review_id,
        SUBSTRING_INDEX(
            target_fully_qualified_name, '.', -1
        )                                              AS target_table,
        review_timestamp,
        overall_status,
        total_checks_run,
        total_checks_passed,
        total_checks_warned,
        total_checks_critical,
        ai_failure_category,
        ROUND(ai_confidence_score, 2)                  AS ai_confidence
    FROM {REVIEW_RESULTS}
    ORDER BY review_timestamp DESC
    LIMIT 10
""").show(truncate=False)

# Individual check results
print("\n2. Check Results (WARNING and CRITICAL):\n")

for review_record in all_review_records:
    if review_record.get("overall_status") == STATUS_PASS:
        continue

    table = review_record["target_fully_qualified_name"].split(".")[-1]
    print(f"  {table}  —  {review_record['overall_status']}")

    check_results = json.loads(review_record.get("check_results", "[]"))
    for check in check_results:
        status = check.get("check_status", "")
        if status in (STATUS_WARNING, STATUS_CRITICAL):
            emoji = "🔴" if status == STATUS_CRITICAL else "🟡"
            print(f"    {emoji} {check['check_name']} [{check['check_level']}]")
            for finding in check.get("findings", []):
                print(f"       {finding}")
    print()

# AI summaries
print("\n3. AI Summaries:\n")
for review_record in all_review_records:
    if review_record.get("ai_summary"):
        table = review_record["target_fully_qualified_name"].split(".")[-1]
        print(f"  {table}")
        print(f"  Category   : {review_record.get('ai_failure_category')}")
        print(f"  Confidence : {review_record.get('ai_confidence_score')}")
        print(f"  Summary    : {review_record.get('ai_summary')}")
        print(f"  Action     : {review_record.get('ai_recommended_action')}")
        print()

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 3 SQL REVIEW COMPLETE")
print("=" * 60)
print(f"Transformations reviewed : {reviews_done}")
print(f"Transformations failed   : {reviews_failed}")
print(f"Business date checked    : {BUSINESS_DATE}")
print(f"Completed at             : {datetime.now()}")
print()

for review_record in all_review_records:
    table  = review_record["target_fully_qualified_name"].split(".")[-1]
    status = review_record.get("overall_status", "UNKNOWN")
    emoji  = "✅" if status == "PASS" else "🔴" if status == "CRITICAL" else "🟡"
    print(
        f"  {emoji} {table:<40} "
        f"Status: {status}  "
        f"Checks: {review_record['total_checks_run']} run, "
        f"{review_record['total_checks_passed']} passed"
    )

print()
print("=" * 60)